# Single-particle orbit in a periodic electrostatic potential

This experiment evolves one full-cyclotron particle with classical RK4. The animation shows the particle, its accumulated orbit, and the time-dependent periodic potential.

All potential, physical, initial-condition, and numerical parameters are explicit below so that the result can be reproduced.

In [1]:
%matplotlib inline

import numpy as np

from classes import (
    FullCyclotronDynamics,
    InitialValueProblem,
    RK4,
    SimulationRequest,
    SimulationRunner,
    TrajectoryFC,
)
from workflows import (
    RandomPotentialConfig,
    animate_fc_particle_solution,
    display_animation,
)

## Reproducible configuration

`rho` sets the normalized Larmor radius and `eta` sets the signed cyclotron time scale. The initial velocity coordinates are normalized model coordinates; their physical position-rate scale is reported by `trajectory.velocity_scale`.

In [2]:
potential_config = RandomPotentialConfig(
    amplitude=0.2,
    max_wave_number=8,
    nx=48,
    ny=48,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()

rho = 0.2
eta = 0.1
initial_position = (np.pi, np.pi)
initial_velocity = (1.0, 0.0)

t_span = (0.0, 8 * np.pi)
max_step = 0.002
output_sample_count = 501

trajectory = TrajectoryFC.from_components(
    x=np.asarray([initial_position[0]]),
    y=np.asarray([initial_position[1]]),
    vx=np.asarray([initial_velocity[0]]),
    vy=np.asarray([initial_velocity[1]]),
    rho=rho,
    eta=eta,
)
dynamics = FullCyclotronDynamics(potential, rho=rho, eta=eta)
problem = InitialValueProblem(dynamics, trajectory)
request = SimulationRequest.uniform(
    t_span=t_span,
    max_step=max_step,
    sample_count=output_sample_count,
)

print(potential_config)
print(f"Position-rate scale: {trajectory.velocity_scale:.3f}")

RandomPotentialConfig(amplitude=0.2, max_wave_number=8, nx=48, ny=48, seed=27, interpolation_order=5)
Position-rate scale: 1.000


## Integrate the orbit

In [3]:
solution = SimulationRunner().simulate(problem, RK4(), request)

print(f"Fixed RK4 steps: {solution.diagnostics['step_count']}")

Fixed RK4 steps: 12567


## Animated orbit

The potential and particle state are shown at the same saved times. The black curve is the orbit accumulated up to the current frame.

In [4]:
animation = animate_fc_particle_solution(
    potential,
    solution,
    frames=151,
    interval=80,
    cmap="RdBu_r",
    repeat=True,
)

display_animation(animation, embed_limit_mb=30.0)